<a href="https://colab.research.google.com/github/AyanBhardwaj1/LowResourceResearch/blob/main/AyanbelarusianQA_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BelarusianQA

## HouseKeeping

In [ ]:
!pip install ipython-autotime
!pip install openai==0.27.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.107.0
    Uninstalling openai-1.107.0:
      Successfully uninstalled openai-1.107.0


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
%load_ext autotime
drive.mount('/content/gdrive')

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
time: 433 ms (started: 2025-09-16 22:53:38 +00:00)


## API Keys

In [ ]:
# API access keys
api_keys = pd.read_csv('/content/gdrive/MyDrive/Computing/api_keys.csv')
api_keys = pd.DataFrame(api_keys)
openai_api_key = api_keys['key'][api_keys['api'] =='openai'].iloc[0]
claude_api_key = api_keys['key'][api_keys['api'] =='claude'].iloc[0]
deepseek_api_key = api_keys['key'][api_keys['api'] =='deepseek'].iloc[0]
gemini_api_key = api_keys['key'][api_keys['api'] =='gemini'].iloc[0]


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/Computing/api_keys.csv'

time: 10.3 ms (started: 2025-09-16 22:52:44 +00:00)


## Data

In [ ]:
df = pd.read_csv("/content/gdrive/MyDrive/BelarusianQA/data/belwiki_500_paragraphs_topic_balanced.csv")
df.shape

FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/BelarusianQA/data/belwiki_500_paragraphs_topic_balanced.csv'

time: 7.8 ms (started: 2025-09-16 22:52:34 +00:00)


In [ ]:
!pip install langdetect

from langdetect import detect, DetectorFactory
import pandas as pd

# Ensure consistent results by setting a seed
DetectorFactory.seed = 0


# Detect language for each paragraph
def detect_language(text):
    try:
        return detect(str(text))
    except:
        return 'unknown'

df['language'] = df['paragraph'].apply(detect_language)

language_counts = df['language'].value_counts()

# Display results
print(language_counts)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=ec9fa95e838f3613e6c894889034c495dc9725aea6773b72b5f67b41cdb940f5
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


NameError: name 'df' is not defined

time: 8.95 s (started: 2025-09-16 22:51:45 +00:00)


## GPT-4.1-mini

In [ ]:
import openai
import time
import uuid
import json

# Set up your OpenAI API key
openai.api_key = openai_api_key
MODEL = "gpt-4.1-mini"

PROMPT_TEMPLATE = """You are a careful Belarusian QA writer.
Given the following Belarusian paragraph, write 2-3 question–answer pairs.
Rules:
- Both question and answer must be in Belarusian.
- The answer must come only from the paragraph.
- Keep answers short and factual.
- Return strict JSON: {{"qas":[{{"question":"...","answer":"..."}}, ...]}}

Paragraph:
\"\"\"{context}\"\"\"
"""

def _extract_json_block(text: str) -> str | None:
    """
    Best-effort: pull out the first top-level JSON object from text.
    Helps when the model wraps JSON with extra text despite instructions.
    """
    # Find the first {...} block
    start = text.find("{")
    if start == -1:
        return None
    # Simple stack-based scan to match braces
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def _safe_parse_json(raw: str) -> dict | None:
    """Parse JSON, with a fallback that extracts the first JSON object from text."""
    try:
        return json.loads(raw)
    except Exception:
        block = _extract_json_block(raw)
        if block:
            try:
                return json.loads(block)
            except Exception:
                return None
        return None

def generate_qa_with_openai(paragraph: str, model: str = MODEL):
    """
    Call OpenAI once for a single paragraph and return up to 3 QA pairs.
    Always returns a list (possibly empty).
    """
    prompt = PROMPT_TEMPLATE.format(context=str(paragraph).strip())

    resp = openai.ChatCompletion.create(
        model=model,
        messages=[
            # Strong JSON-only instructions
            {"role": "system", "content": "You are a JSON generator. Return ONLY valid JSON. No text outside JSON."},
            {"role": "system", "content": "The JSON must have a top-level key 'qas' with a list of objects."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content

    data = _safe_parse_json(raw)
    if not isinstance(data, dict):
        print("Could not parse JSON. Raw output:\n", raw)
        return []

    qas = data.get("qas", [])
    if not isinstance(qas, list):
        print("'qas' missing or not a list. Parsed:\n", data)
        return []

    # Normalize and keep up to 3
    out = []
    for qa in qas[:3]:
        if isinstance(qa, dict):
            q = str(qa.get("question", "")).strip()
            a = str(qa.get("answer", "")).strip()
            if q and a:
                out.append({"question": q, "answer": a})
    return out


# Work on the first 3 rows
sample_df = df.head(3).copy()

results = []
for _, row in sample_df.iterrows():
    qas = generate_qa_with_openai(row["paragraph"])
    for qa in qas:
        results.append({
            "page_id": row["page_id"],
            "title": row["title"],
            "paragraph": row["paragraph"],
            "question": qa["question"],
            "answer": qa["answer"],
            "gen_uid": str(uuid.uuid4())
        })
    time.sleep(0.25)  # light pacing

qa_df = pd.DataFrame(results)

# --- Save JSONL (one object per line) ---
with open("out.jsonl", "w", encoding="utf-8") as f:
    for rec in qa_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# --- Save pretty JSON array (human-friendly) ---
with open("out.json", "w", encoding="utf-8") as f:
    json.dump(qa_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# --- Print a beautified preview (first 3 records) ---
print("Generated", len(qa_df), "QA pairs from", len(sample_df), "paragraphs")
print(json.dumps(qa_df.to_dict(orient="records")[:3], ensure_ascii=False, indent=2))

NameError: name 'openai_api_key' is not defined

time: 7.56 ms (started: 2025-09-16 22:52:08 +00:00)


## Claude